# International Open LiDAR Data Guide

This notebook lists freely available LiDAR datasets worldwide and shows
how to load each into Occulus.  All datasets are public domain or open licence.

## United States

| Source | Coverage | Format | URL |
|---|---|---|---|
| **KY From Above** (AbovePy) | All 120 KY counties | LAZ/COPC | kyfromabove.ky.gov |
| **USGS 3DEP** | Nationwide | LAS/LAZ | nationalmap.gov |
| **NOAA Coastal** | US coastlines | LAS/LAZ | coast.noaa.gov/dataviewer |
| **OpenTopography** | Worldwide | LAS/LAZ | opentopography.org |

## Europe

| Source | Coverage | Format | URL |
|---|---|---|---|
| **AHN** (Netherlands) | All Netherlands | LAZ | ahn.nl |
| **UK Environment Agency** | England | LAZ | environment.data.gov.uk |
| **French IGN LiDAR HD** | France | COPC/LAZ | geoservices.ign.fr |
| **Swisstopo** | Switzerland | LAS | swisstopo.admin.ch |

## Rest of World

| Source | Coverage | Format | URL |
|---|---|---|---|
| **LINZ** | New Zealand | LAZ | data.linz.govt.nz |
| **ELVIS** | Australia | LAZ | elevation.fsdf.org.au |
| **GeoBasis-DE** | Germany (state-by-state) | LAS | varies by state |
| **OpenTopography** | Global (research datasets) | LAS/LAZ | opentopography.org |

In [ ]:
# KY From Above via AbovePy (if installed)
try:
    import abovepy
    from abovepy import search
    
    print('=== Kentucky From Above (AbovePy) ===')
    tiles = search('lidar', county='pike', phase=3)  # Pike County, eastern KY
    print(f'Pike County Phase 3 tiles: {len(tiles)}')
    if tiles:
        print(f'  First tile: {tiles[0]}')
except ImportError:
    print('AbovePy not installed. Install: pip install abovepy[lidar]')

In [ ]:
# USGS 3DEP — direct download
import urllib.request, tempfile
from pathlib import Path

# Small Oregon forest tile
URL = (
    'https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/LPC/Projects/'
    'USGS_LPC_OR_WillametteNF_2018/laz/USGS_LPC_OR_WillametteNF_2018_e0555n4855.laz'
)

cache = Path(tempfile.gettempdir()) / 'occulus_intl_demo'
cache.mkdir(exist_ok=True)
dest = cache / Path(URL).name

if not dest.exists():
    print(f'Downloading Oregon tile (~3 MB)…')
    try:
        urllib.request.urlretrieve(URL, str(dest))
        print('Done.')
    except Exception as e:
        print(f'Download failed: {e}')
else:
    print(f'Cached: {dest.name}')

In [ ]:
# Load and analyse
if dest.exists():
    from occulus.io import read
    from occulus.metrics import compute_cloud_statistics
    
    cloud = read(dest, platform='aerial', subsample=0.2)
    stats = compute_cloud_statistics(cloud)
    print(f'Oregon Willamette NF: {cloud.n_points:,} points')
    print(f'Z range: {stats.z_min:.1f} – {stats.z_max:.1f} m')
    print(f'Vertical relief: {stats.z_max - stats.z_min:.0f} m')

## Netherlands AHN3

```python
# AHN tiles can be fetched from GeoTiles (TU Delft)
# https://geotiles.citg.tudelft.nl/
# Tile naming: r{col}_{row}.laz
import urllib.request
url = 'https://geotiles.citg.tudelft.nl/AHN3_t/37ez1.LAZ'
urllib.request.urlretrieve(url, 'ahn3_37ez1.laz')

from occulus.io import read
cloud = read('ahn3_37ez1.laz', platform='aerial')
print(cloud)  # note: Z referenced to NAP (Dutch ordnance datum)
```

## UK Environment Agency

```python
# EA 1m LiDAR Composite (England)
# https://environment.data.gov.uk/dataset/3e6bd94a-2245-4963-8b8e-88e5c13e41d6
# Download tiles as GeoTIFF or LAZ from the data portal
```

## French IGN LiDAR HD

```python
# https://geoservices.ign.fr/lidarhd
# 5 pts/m² nationwide coverage, free download (sign-in required)
# Tiles in LAS 1.4 format
```

## New Zealand LINZ

```python
# https://data.linz.govt.nz/data/category/elevation/
# Search 'LiDAR 1m DEM' — LAZ downloads are free
```

In [ ]:
# Summary: coverage by region
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

regions = [
    ('KY From Above', 'United States', 'steelblue', 'Free, 1m NPS, AbovePy API'),
    ('USGS 3DEP', 'United States', 'steelblue', 'Free, 0.5–2m NPS'),
    ('NOAA Coastal', 'United States', 'steelblue', 'Free, coastal areas'),
    ('Netherlands AHN4', 'Europe', 'forestgreen', 'Free, 0.5m NPS, NAP datum'),
    ('UK EA 1m LiDAR', 'Europe', 'forestgreen', 'Free, England only'),
    ('French IGN LiDAR HD', 'Europe', 'forestgreen', 'Free (login), 5 pts/m²'),
    ('LINZ NZ', 'Pacific', 'purple', 'Free, 1–4 pts/m²'),
    ('ELVIS Australia', 'Pacific', 'purple', 'Free, variable coverage'),
    ('OpenTopography', 'Global', 'tomato', 'Free (API key), research datasets'),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
col_headers = ['Dataset', 'Region', 'Access', 'Notes']
table_data = [(r[0], r[1], '✓ Free', r[3]) for r in regions]
tbl = ax.table(
    cellText=table_data,
    colLabels=col_headers,
    cellLoc='left',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#ecf0f1')

ax.set_title('Open LiDAR Data Sources — Global Overview', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig('../../outputs/international_data_guide.png', dpi=150, bbox_inches='tight')
plt.show()

**Next**: `10_change_detection.ipynb`